# Flood Segmentation — Inference from Best Checkpoint
This notebook loads the best checkpoint saved by the training notebook
(`aisehack_fardin_composite_focal_diceloss.ipynb`) and runs validation,
prediction, and submission CSV generation — **no re-training required**.

In [ ]:
# ── Optional installs (uncomment if needed) ───────────────────────────────────
# !pip install git+https://github.com/IBM/terratorch.git
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 --force-reinstall

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor
from typing import Any
from pathlib import Path

import terratorch
from terratorch.tasks import SemanticSegmentationTask

import albumentations
import lightning.pytorch as pl
import rasterio
import pandas as pd

## 1 · Configuration — update these paths before running

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATASET_PATH = '/kaggle/input/datasets/manipes/ibm-flood-dataset-phase-2/data/'

# Path to the best .ckpt file produced by the training notebook.
# Typically found at: <OUT_DIR>/test/checkpoints/best-checkpoint-*.ckpt
CHECKPOINT_PATH = '/kaggle/working/test/checkpoints/best-checkpoint-epoch=00-val_loss=0.00.ckpt'

# Output directory for predictions and submission CSV
OUTPUT_DIR   = '/kaggle/working/inference'
PRED_DIR     = os.path.join(OUTPUT_DIR, 'prediction')
SUBMISSION_CSV = os.path.join(OUTPUT_DIR, 'submission.csv')

os.makedirs(PRED_DIR, exist_ok=True)

# ── Hyperparameters (must match training) ─────────────────────────────────────
BANDS         = [1, 2, 3, 4, 5, 6]
NUM_FRAMES    = 1
CLASS_WEIGHTS = [0.51, 2.19, 1.71]
HEAD_DROPOUT  = 0.2
LR            = 1.0e-4
WEIGHT_DECAY  = 0.05

print(f'Checkpoint : {CHECKPOINT_PATH}')
print(f'Predictions: {PRED_DIR}')

## 2 · Re-define the custom loss classes
These must be present in the Python environment so the checkpoint can be
restored correctly (Lightning stores the class reference in the checkpoint).

In [ ]:
# ── Focal Loss ─────────────────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, class_weights, ignore_index=-1, gamma=2.0):
        super().__init__()
        self.ignore_index = ignore_index
        self.gamma        = gamma
        self.register_buffer('class_weights', torch.tensor(class_weights, dtype=torch.float32))

    def forward(self, logits: Tensor, targets: Tensor) -> Tensor:
        ce = F.cross_entropy(
            logits, targets,
            weight=self.class_weights.to(logits.device),
            ignore_index=self.ignore_index,
            reduction='none',
        )
        probs        = F.softmax(logits, dim=1)
        valid_mask   = targets != self.ignore_index
        safe_targets = targets.clone()
        safe_targets[~valid_mask] = 0
        pt           = probs.gather(1, safe_targets.unsqueeze(1)).squeeze(1)
        focal_weight = (1 - pt) ** self.gamma
        return (focal_weight * ce)[valid_mask].mean()


# ── Multiclass Dice Loss ───────────────────────────────────────────────────────
class MulticlassDiceLoss(nn.Module):
    def __init__(self, ignore_index=-1, smooth=1e-6):
        super().__init__()
        self.ignore_index = ignore_index
        self.smooth       = smooth

    def forward(self, logits: Tensor, targets: Tensor) -> Tensor:
        n_classes    = logits.shape[1]
        probs        = F.softmax(logits, dim=1)
        valid_mask   = (targets != self.ignore_index)
        safe_targets = targets.clone()
        safe_targets[~valid_mask] = 0
        one_hot = F.one_hot(safe_targets, n_classes).permute(0, 3, 1, 2).float()
        one_hot *= valid_mask.unsqueeze(1).float()
        probs   *= valid_mask.unsqueeze(1).float()
        dims           = (0, 2, 3)
        intersection   = (probs * one_hot).sum(dim=dims)
        cardinality    = (probs + one_hot).sum(dim=dims)
        dice_per_class = 1 - (2 * intersection + self.smooth) / (cardinality + self.smooth)
        return dice_per_class.mean()


# ── Composite Loss ─────────────────────────────────────────────────────────────
class FocalDiceLoss(nn.Module):
    def __init__(self, class_weights, ignore_index=-1, gamma=2.0,
                 focal_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.focal_weight = focal_weight
        self.dice_weight  = dice_weight
        self.focal        = FocalLoss(class_weights, ignore_index, gamma)
        self.dice         = MulticlassDiceLoss(ignore_index)

    def forward(self, logits: Tensor, targets: Tensor) -> Tensor:
        return (
            self.focal_weight * self.focal(logits, targets) +
            self.dice_weight  * self.dice(logits, targets)
        )


# ── Subclassed Task ────────────────────────────────────────────────────────────
class FocalDiceSegmentationTask(SemanticSegmentationTask):
    def __init__(self, *args: Any,
                 gamma: float = 2.0,
                 focal_weight: float = 0.5,
                 dice_weight: float  = 0.5,
                 **kwargs: Any) -> None:
        self.gamma        = gamma
        self.focal_weight = focal_weight
        self.dice_weight  = dice_weight
        super().__init__(*args, **kwargs)

    def configure_losses(self) -> None:
        self.criterion = FocalDiceLoss(
            class_weights=self.hparams.get('class_weights', None),
            ignore_index =self.hparams.get('ignore_index', -1),
            gamma        =self.gamma,
            focal_weight =self.focal_weight,
            dice_weight  =self.dice_weight,
        )

print('Loss classes defined ✓')

## 3 · Build the DataModule

In [ ]:
datamodule = terratorch.datamodules.GenericNonGeoSegmentationDataModule(
    batch_size=2,
    num_workers=2,
    num_classes=3,

    train_data_root=DATASET_PATH + 'image',
    train_label_data_root=DATASET_PATH + 'label',
    val_data_root=DATASET_PATH + 'image',
    val_label_data_root=DATASET_PATH + 'label',
    test_data_root=DATASET_PATH + 'image',
    test_label_data_root=DATASET_PATH + 'label',

    train_split=DATASET_PATH + 'split/train.txt',
    val_split  =DATASET_PATH + 'split/val.txt',
    test_split =DATASET_PATH + 'split/test.txt',

    img_grep  ='*image.tif',
    label_grep='*label.tif',

    train_transform=None,   # no augmentation needed for inference
    val_transform  =None,
    test_transform =None,

    means=[841.1257, 371.6175, 1734.1789, 1588.3142, 1742.8452, 1218.5551],
    stds =[473.7090, 170.3611,  623.0474,  612.8465,  564.5835,  528.0894],
    no_data_replace =0,
    no_label_replace=-1,

    predict_data_root=DATASET_PATH + 'prediction/image/',
)

print('DataModule created ✓')

## 4 · Load model from checkpoint
We use `load_from_checkpoint` which restores architecture **and** weights in
one call — no need to manually call `load_state_dict`.

In [ ]:
# ── Model architecture args (must match training exactly) ──────────────────────
decoder_args = dict(
    decoder='UperNetDecoder',
    decoder_channels=128,
    decoder_scale_modules=True,
)

necks = [
    dict(name='SelectIndices', indices=[2, 5, 8, 11]),
    dict(name='ReshapeTokensToImage', effective_time_dim=NUM_FRAMES),
    dict(name='LearnedInterpolateToPyramidal'),
]

backbone_args = dict(
    backbone_pretrained=True,
    backbone='prithvi_eo_v2_tiny_tl',
    backbone_bands=BANDS,
    backbone_num_frames=1,
)

model_args = dict(
    **backbone_args,
    **decoder_args,
    num_classes=3,
    head_dropout=HEAD_DROPOUT,
    necks=necks,
    rescale=True,
)

# ── Load from checkpoint ───────────────────────────────────────────────────────
model = FocalDiceSegmentationTask.load_from_checkpoint(
    CHECKPOINT_PATH,
    model_args    =model_args,
    map_location  ='cpu',        # move to GPU automatically via Trainer
    strict        =False,        # tolerates minor mismatches (e.g. loss buffers)
    # Pass the same non-hparam constructor kwargs used during training:
    gamma        =2.0,
    focal_weight =0.5,
    dice_weight  =0.5,
)
model.eval()
print('Checkpoint loaded ✓')
print(f'  Source: {CHECKPOINT_PATH}')

## 5 · Build a minimal Trainer (inference only)

In [ ]:
trainer = pl.Trainer(
    accelerator='auto',
    devices    ='auto',
    precision  ='bf16-mixed',
    logger     =False,           # no logging needed
    enable_checkpointing=False,
    tiled_inference_on_validation=True,
    tiled_inference_on_testing   =True,
    tiled_inference_parameters={
        'h_crop':    256,
        'w_crop':    256,
        'h_stride':  192,
        'w_stride':  192,
        'batch_size': 2,
    },
)
print('Trainer ready ✓')

## 6 · Validate on the validation split

In [ ]:
datamodule.setup('fit')   # sets up val_dataset

val_results = trainer.validate(model, datamodule=datamodule)
print('\nValidation results:')
for k, v in val_results[0].items():
    print(f'  {k}: {v:.4f}')

## 7 · Run on the test split

In [ ]:
datamodule.setup('test')

test_results = trainer.test(model, datamodule=datamodule)
print('\nTest results:')
for k, v in test_results[0].items():
    print(f'  {k}: {v:.4f}')

## 8 · Predict on the unlabelled prediction split and save GeoTIFFs

In [ ]:
datamodule.setup('predict')

predictions = trainer.predict(model, datamodule=datamodule)

for batch_idx, (preds, file_paths) in enumerate(predictions):
    if isinstance(preds, tuple):
        preds = preds[0]

    if preds.ndim == 4:               # [B, C, H, W] → argmax
        preds = preds.argmax(dim=1)   # [B, H, W]

    preds = preds.cpu().numpy().astype('int16')

    for i in range(preds.shape[0]):
        arr = preds[i]
        arr[arr < 0] = -1             # enforce ignore_index

        ref_path = file_paths[i]
        with rasterio.open(ref_path) as src:
            meta = src.meta.copy()

        meta.update({'count': 1, 'dtype': 'int16', 'nodata': -1, 'compress': 'lzw'})

        out_name = os.path.splitext(os.path.basename(ref_path))[0] + '.tif'
        out_path = os.path.join(PRED_DIR, out_name)

        with rasterio.open(out_path, 'w', **meta) as dst:
            dst.write(arr, 1)

        print(f'Saved {out_path}')

print('\nAll predictions saved ✓')

## 9 · Generate Kaggle submission CSV

In [ ]:
def mask_to_rle(mask: np.ndarray) -> str:
    if mask.sum() == 0:
        return '0 0'
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs   = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)


def rle_to_mask(rle_str: str, shape) -> np.ndarray:
    mask    = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    rle_str = str(rle_str).strip()
    if rle_str in ('0 0', '', '-1', 'nan'):
        return mask.reshape(shape, order='F')
    parts = list(map(int, rle_str.split()))
    for start, length in zip(parts[0::2], parts[1::2]):
        mask[start - 1: start - 1 + length] = 1
    return mask.reshape(shape, order='F')


def generate_submission_csv(pred_dir: str, output_csv: str):
    pred_dir  = Path(pred_dir)
    tif_files = sorted(pred_dir.glob('*.tif'))

    if not tif_files:
        print(f'No prediction TIFs found in {pred_dir}')
        return None

    rows = []
    for tif_path in tif_files:
        with rasterio.open(tif_path) as src:
            pred = src.read(1)

        flood_mask = (pred == 1).astype(np.uint8)
        rle        = mask_to_rle(flood_mask)

        if not rows:  # round-trip check on first image
            decoded = rle_to_mask(rle, flood_mask.shape)
            status  = 'PASS' if np.array_equal(flood_mask, decoded) else 'FAIL'
            print(f'RLE round-trip check: {status}')

        image_id = tif_path.name.replace('_image.tif', '')
        rows.append({'id': image_id, 'rle_mask': rle})

    df = pd.DataFrame(rows, columns=['id', 'rle_mask'])
    df['rle_mask'] = df['rle_mask'].replace('', '0 0').fillna('0 0')
    df.to_csv(output_csv, index=False)

    n_flood    = (df['rle_mask'] != '0 0').sum()
    n_no_flood = (df['rle_mask'] == '0 0').sum()
    print(f'\nSubmission CSV saved  : {output_csv}')
    print(f'Total patches         : {len(df)}')
    print(f'With flood pixels     : {n_flood}  ({100*n_flood/len(df):.1f}%)')
    print(f'No flood ("0 0")      : {n_no_flood}  ({100*n_no_flood/len(df):.1f}%)')
    print(df.head(5).to_string(index=False))
    return df


df_sub = generate_submission_csv(PRED_DIR, SUBMISSION_CSV)

## Done!
The submission CSV is ready at `SUBMISSION_CSV`.
Submit it directly to the Kaggle competition page.